<a href="https://colab.research.google.com/github/Dan-Z-26/DSC---Recruitment-Data-Science---Task/blob/main/DSC_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =================================================================
# PART 1: DATA CLEANING & SCORING
# =================================================================

# 1. Load Data
df = pd.read_csv('/content/final_influencer_hunt_dirty.csv')
pd.set_option('display.max_columns', None)

# 2. Cleaning Functions
def clear_followers(follower_str):
    if pd.isna(follower_str): return np.nan
    f_str = str(follower_str).strip().lower().replace(',', '')
    if 'k' in f_str: return int(float(f_str.replace('k', ''))) * 1000
    if 'm' in f_str: return int(float(f_str.replace('m', ''))) * 1000000
    return int(float(f_str))

df['Followers'] = df['Followers'].apply(clear_followers).astype('Int64')

# 3. Handle Missing Values
for col in df.select_dtypes(include=np.number).columns[df.select_dtypes(include=np.number).isnull().any()]:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns[df.select_dtypes(include='object').isnull().any()]:
    df[col] = df[col].fillna('Unknown')

# 4. Filter & Deduplicate
df = df.drop_duplicates(subset=['IG_Handle']).copy()
df_v = df[(df['Followers'] >= 1000) & (df['Followers'] <= 1000000) & (df['Email_Status'] == 'Verified')].copy()

# 5. Feature Engineering
df_v['Eng_Rate'] = (df_v['Eng_Auth'] / df_v['Followers'].astype(float)) * 100
tech_map = ['Tech/AI', 'Coding', 'AI/ML', 'Data Science', 'Cloud Computing', 'Tech', 'Software', 'AI']
df_v['Main_Category'] = df_v['Category'].apply(lambda x: 'Tech' if x in tech_map else x)

# 6. Benchmark Scoring
def benchmark_score(series):
    benchmark = series.quantile(0.90)
    if benchmark == 0 or pd.isna(benchmark): benchmark = series.max()
    return (series / benchmark).clip(0, 1) * 100

df_v['Reach_Score'] = benchmark_score(df_v['Avg_Likes'])
df_v['Activity_Score'] = benchmark_score(df_v['Eng_Rate'])
df_v['Consistency_Score'] = benchmark_score(df_v['Posts_Per_Week'])

# Final Weighted Score
df_v['Final_Score'] = ((df_v['Activity_Score'] * 0.5) + (df_v['Reach_Score'] * 0.3) + (df_v['Consistency_Score'] * 0.2)).round(1)

print("DATA CLEANING & SCORING COMPLETE")
print(f"Total Micro-Influencers Ranked: {len(df_v)}")


# =================================================================
# PART 2: DATA VISUALIZATION
# =================================================================

print("\n--- PERFORMANCE RANKINGS BY CATEGORY ---")
sns.set_style("whitegrid")
unique_cats = sorted([c for c in df_v['Main_Category'].unique() if c != 'Unknown'])

for cat in unique_cats:
    top_5 = df_v[df_v['Main_Category'] == cat].sort_values('Final_Score', ascending=False).head(5)
    if not top_5.empty:
        plt.figure(figsize=(9, 4))
        ax = sns.barplot(data=top_5, x='Final_Score', y='Influencer_Name', hue='Influencer_Name', palette="viridis", legend=False)
        for container in ax.containers: ax.bar_label(container, padding=3, fmt='%.1f%%')

        plt.title(f'Top 5 {cat} Performers', fontsize=12, fontweight='bold')
        plt.xlabel('Final Score (%)')
        plt.ylabel('')
        plt.xlim(0, 115)
        sns.despine(left=True, bottom=True)
        plt.tight_layout()
        plt.show()

        print(f"\n {cat.upper()} LEADERBOARD :")
        for _, row in top_5.iterrows():
            print(f"- {row['Influencer_Name']:<20} | Score: {row['Final_Score']:>5} | Audience Activity: {int(row['Activity_Score']):>3} | Reach: {int(row['Reach_Score']):>3} | Consistency: {int(row['Consistency_Score']):>3}\n")
        print("-" * 100,'\n')